In [ ]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

In [ ]:
import numpy as np
import pickle
import os, yaml

from matplotlib import pyplot as plt
from pylab import *

In [ ]:
import sys
sys.path.append('./code')

In [ ]:
ROOT_PATH = './'

In [ ]:
GRID=np.linspace(8900, 9050, 1000)
energy_grid = GRID[333:733:2]

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
from torch.utils.data import DataLoader
from models import XASLightningModule
from data import XASGraphDataset, collate_fn

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
with open(ROOT_PATH + "/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
nblocks, cutoff, threebody_cutoff = 3, 4.0, 4.0
r1, r2, rZn = 3.0, 5.0, 4.2

In [ ]:
SHIFT = 705+0.3
scale_predicted = 19.84 / 20

In [ ]:
element_types=("O", "H", "Zn", "Cl")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from PIL import Image

### 1. Load trained model

In [ ]:
gnn_config = dict(
    nblocks = config['gnn']['nblocks'], 
    cutoff = config['gnn']['cutoff'], 
    threebody_cutoff = config['gnn']['threebody_cutoff']
)

head_config = dict(
    hidden_dims = config['head']['hidden_dims'], 
    output_size = config['head']['output_size'], 
    drop_rate = config['head']['drop_rate'], 
)

model = XASLightningModule(gnn_config, head_config, learning_rate=config['training']['lr'])

model.load_state_dict(torch.load(os.path.join('./model', "full_model.pth"), weights_only=True))

model = model.to(device)
model.eval()

### 2. Decile plots of test dataset

In [ ]:
test_structures = pickle.load(open(ROOT_PATH + '/dataset/test_structures.pkl', 'rb'))
test_spectra = torch.load(ROOT_PATH + "/dataset/test_spectra.pt")

In [ ]:
test_dataset = XASGraphDataset(test_structures, test_spectra, cutoff=cutoff)
test_loader = DataLoader(test_dataset, batch_size=config['training']['batch_size'], collate_fn=collate_fn)

In [ ]:
predictions = []
test_loss = 0.0

with torch.no_grad(): 
    for g, _, spectra in test_loader:
        g, spectra = g.to(device), spectra.to(device)
        y_hat = model(g)

        predictions.append(y_hat.cpu())

        loss = torch.nn.MSELoss()(y_hat, spectra)
        test_loss += loss.item() * spectra.size(0)  

# concatenate all predictions into one tensor
predictions = torch.cat(predictions, dim=0)

print(f"Test loss: {test_loss / len(test_loader.dataset)}")

In [ ]:
test_performance = []
for ii, data in enumerate(test_dataset): 
    _, _, spectrum = data
    test_performance.append((nn.MSELoss()(spectrum, predictions[ii]).item(), ii))

test_performance.sort()

In [ ]:
dict_decile_test = {}

idx = [i for loss, i in [test_performance[j] for j in np.arange(0+14, len(test_performance)+14, len(test_performance)//10) ]]
for top_i, i in enumerate(idx):
    pred = predictions[i]
    _, _, sp = test_dataset[i]
    loss = nn.MSELoss()(sp, pred).item()
    dict_decile_test[top_i] = {
        'idx': i, 
        'spectrum': sp.numpy(), 
        'prediction': pred.numpy(), 
        'loss': loss
    }
    plt.plot(energy_grid, pred.numpy(), label='Predicted')
    plt.plot(energy_grid, sp.numpy(), label='True')
    plt.annotate(f'MSE = {loss:.3e}', xy=(8980, 0.1), fontsize=12)
    
    plt.legend()
    # plt.savefig('./figures/gnn-128_128-prediction-%s.png'%(top_i+1), bbox_inches='tight', dpi=80)
    plt.show()


In [ ]:
# pickle.dump(dict_decile_test, open('./data/test_dataset-decile.pkl', 'wb'), protocol=4)

### 3. Node Importance

In [ ]:
import dgl
import plotly.graph_objects as go

In [ ]:
def wrap_to_nearest_image(positions, center, lattice_matrix):

    # Convert to fractional coordinates
    lattice_inv = np.linalg.inv(lattice_matrix)
    frac_positions = positions @ lattice_inv
    frac_center = center @ lattice_inv
    
    # Calculate displacement in fractional coordinates
    frac_displacements = frac_positions - frac_center
    
    # Wrap to nearest image: shift by -1, 0, or +1 unit cells
    frac_displacements = frac_displacements - np.round(frac_displacements)
    
    # Convert back to Cartesian
    wrapped_displacements = frac_displacements @ lattice_matrix
    wrapped_positions = center + wrapped_displacements
    
    return wrapped_positions

In [ ]:
structures = pickle.load(open(ROOT_PATH + '/dataset/all_structures.pkl', 'rb'))
spectra = torch.load(ROOT_PATH + "/dataset/all_spectra.pt")

In [ ]:
dataset = XASGraphDataset(structures, spectra, cutoff=cutoff)
torch.save({
    "graphs": [g for g, _, _ in dataset],
    "states": [s for _, s, _ in dataset],
    "spectra": [y for _, _, y in dataset]
}, ROOT_PATH + "/tmp/all_data-%s.pt"%cutoff)

In [ ]:
all_data = torch.load(ROOT_PATH + "/tmp/all_data-%s.pt"%cutoff)

In [ ]:
def plot_wrapped_subgraph_3d_interactive(structure, subg, node_sizes, cutoff_distance=3.0,
                                         element_types=("O", "H", "Zn", "Cl")):
    """Create interactive 3D plot with Plotly."""
    # Color and size maps
    color_map = {0: 'red', 1: 'lightblue', 2: 'gray', 3: 'green'}
    cutoff_dict = {
        ('Zn', 'O'): 3.0,
        ('Zn', 'Cl'): 3.0,
        ('Zn', 'H'): 2.0,
        ('O', 'H'): 1.5,
        ('H', 'H'): 1.2,
        ('O', 'O'): 2.0,
        ('Cl', 'Cl'): 2.5,
        ('O', 'Cl'): 2.0,
        ('Cl', 'H'): 1.5,
    }
    node_type_ids = subg.ndata['node_type'].cpu().numpy()
    
    # Get positions and wrap them
    center_pos = subg.ndata['pos'][0].cpu().numpy()
    pos_3d = subg.ndata['pos'].cpu().numpy()
    lattice_matrix = structure.lattice.matrix
    
    wrapped_pos_3d = wrap_to_nearest_image(pos_3d, center_pos, lattice_matrix)
    wrapped_pos_3d -= center_pos
    
    # Create filtered edges
    src, dst = subg.edges()
    edges = []
    for i in range(len(src)):
        s, d = src[i].item(), dst[i].item()
        
        # Get element types for both nodes
        elem_s = element_types[node_type_ids[s]]
        elem_d = element_types[node_type_ids[d]]
        
        # Get cutoff for this pair (check both orders)
        cutoff = cutoff_dict.get((elem_s, elem_d), 
                                 cutoff_dict.get((elem_d, elem_s), 3.0))
        
        # Calculate distance and check cutoff
        dist = np.linalg.norm(wrapped_pos_3d[s] - wrapped_pos_3d[d])
        if dist < cutoff:
            edges.append((s, d))
    

    # Create lists of colors and sizes for all nodes
    node_colors = [color_map[node_type_ids[i]] for i in range(subg.num_nodes())]
    node_labels = [f'{element_types[node_type_ids[i]]}{i}' for i in range(subg.num_nodes())]
    hover_texts = [f'{element_types[node_type_ids[i]]}{i}<br>Pos: ({wrapped_pos_3d[i,0]:.2f}, {wrapped_pos_3d[i,1]:.2f}, {wrapped_pos_3d[i,2]:.2f})'
                   for i in range(subg.num_nodes())]
    
    # Create edge traces
    edge_traces = []
    for s, d in edges:
        edge_trace = go.Scatter3d(
            x=[wrapped_pos_3d[s, 0], wrapped_pos_3d[d, 0], None],
            y=[wrapped_pos_3d[s, 1], wrapped_pos_3d[d, 1], None],
            z=[wrapped_pos_3d[s, 2], wrapped_pos_3d[d, 2], None],
            mode='lines',
            line=dict(color='gray', width=3),
            hoverinfo='none',
            showlegend=False
        )
        edge_traces.append(edge_trace)
    
    # Create single node trace with color list
    node_trace = go.Scatter3d(
        x=wrapped_pos_3d[:, 0],
        y=wrapped_pos_3d[:, 1],
        z=wrapped_pos_3d[:, 2],
        # mode='markers+text',
        mode='markers',
        text=node_labels,
        textposition='top center',
        marker=dict(
            size=node_sizes,
            color=node_colors,
            # line=dict(color='black', width=2)
            line=dict(width=0)
        ),
        # hovertext=hover_texts,
        # hoverinfo='text',
        showlegend=False
    )
    
    # Combine all traces
    fig = go.Figure(data=edge_traces + [node_trace])

    dict_common = dict(
                title='', 
                range=[-4.5, 4.5], 
                tickvals=[-4, -3, -2, -1, 0, 1, 2, 3, 4],
                backgroundcolor="white",
                gridcolor="lightgray",
                zeroline=False, 
                showticklabels=False,  
                showspikes=False
    )
    # Update layout
    fig.update_layout(
        # title='Interactive 3D Coordination Environment',
        scene=dict(
            xaxis = dict_common, 
            yaxis = dict_common, 
            zaxis = dict_common, 
            aspectmode='cube'  # or 'cube' for equal aspect ratio
        ),
        paper_bgcolor='white',
        plot_bgcolor='white',                 
        width=1000,
        height=1000
    )  
    return fig


In [ ]:
cluster_reps = [1955, 923, 1878, 1210, 1858, 976]

In [ ]:
threshold_percent = 0.10

In [ ]:
for cluster_idx in range(1): 
    sp_idx = cluster_reps[cluster_idx]
    g = all_data["graphs"][sp_idx].to("cuda")
    structure = structures[sp_idx]
    first_neighbors = structure.get_neighbors(structure[0], r=r1)  # Using 3.0 Angstrom cutoff
    fn_idxs_noH = [int(site.index) for site in first_neighbors if site.specie.symbol != 'H']  # First shell ligands

    node_importance_0baseline_arr = np.loadtxt('./data/data-IG/0_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff))
    
    mean_attr = node_importance_0baseline_arr.mean(axis=0)
    mean_attr_nmlz = mean_attr / np.max(mean_attr[1:])
    # mean_attr_nmlz = mean_attr / np.max(mean_attr)
    mean_attr_nmlz_fn_noH = mean_attr_nmlz[fn_idxs_noH].mean()
    selected =  mean_attr_nmlz > threshold_percent * mean_attr_nmlz_fn_noH
    
    size_arr = sqrt(mean_attr_nmlz[selected])
    sub_nodes = np.arange(len(mean_attr))[selected]
    subg = dgl.node_subgraph(g, sub_nodes)
    
    # Usage
    fig = plot_wrapped_subgraph_3d_interactive(structure, subg, size_arr*30, cutoff_distance=3.0)
    fig.show()
    # fig.write_html('./figures/Node_attr-withBKG-cluster_%s-structure_%s-%s.html'%(cluster_idx, sp_idx, threshold_percent))
    # fig.write_image('./figures/Node_attr-cluster_%s-structure_%s.pdf'%(cluster_idx, sp_idx))


In [ ]:
close('all')
fig = plot_wrapped_subgraph_3d_interactive(structure, subg, size_arr*40, cutoff_distance=3.0)
fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(size=15, color='red'),
    legendgroup='O', showlegend=True, name='O'
))
fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(size=15, color='lightblue'),
    legendgroup='H', showlegend=True, name='H'
))
fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(size=15, color='gray'),
    legendgroup='Zn', showlegend=True, name='Zn'
))
fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(size=15, color='green'),
    legendgroup='Cl', showlegend=True, name='Cl'
))    
# fig.write_html('./figures/OHZnCl_node_legend.html')

### 4. Plot graph representation illustration

In [ ]:
import networkx as nx

In [ ]:
structures = pickle.load(open(ROOT_PATH + '/dataset/all_structures.pkl', 'rb'))
spectra = torch.load(ROOT_PATH + "/dataset/all_spectra.pt")

In [ ]:
all_data = torch.load(ROOT_PATH + "/tmp/all_data-%s.pt"%cutoff)

In [ ]:
sp_idx = 50
g = all_data["graphs"][sp_idx].to("cuda")
structure = structures[sp_idx]

In [ ]:
pred_spectrum_cpu = np.loadtxt(ROOT_PATH+"/data/illustration_spec-pred-structure_%s-%s-%s-%s.txt"%(sp_idx, nblocks, cutoff, threebody_cutoff), unpack=True)[1]

In [ ]:
def plot_2d_graph_for_workflow(subg, structure, 
                               element_types=("O", "H", "Zn", "Cl"),
                               cutoff_dict=None,
                               figsize=(8, 8)):
    
    # Default cutoffs
    if cutoff_dict is None:
        cutoff_dict = {
            ('Zn', 'O'): 4.0, ('Zn', 'Cl'): 4.0, 
            ('Zn', 'H'): 2.0, ('O', 'H'): 1.5,
        }        
    
    # Get node info
    node_type_ids = subg.ndata['node_type'].cpu().numpy()
    
    # Create networkx graph with filtered edges
    G = nx.Graph()
    G.add_nodes_from(range(subg.num_nodes()))
    
    # Filter edges by distance
    pos_3d = subg.ndata['pos'].cpu().numpy()
    src, dst = subg.edges()
    
    for i in range(len(src)):
        s, d = src[i].item(), dst[i].item()
        elem_s = element_types[node_type_ids[s]]
        elem_d = element_types[node_type_ids[d]]
        cutoff = cutoff_dict.get((elem_s, elem_d), 
                                 cutoff_dict.get((elem_d, elem_s), 3.0))
        dist = np.linalg.norm(pos_3d[s] - pos_3d[d])
        if dist < cutoff:
            G.add_edge(s, d)
    
    pos = nx.kamada_kawai_layout(G)
    
    # Color and size settings
    color_map = {0: '#fe0300', 1: '#ffcccc', 2: '#8f8f81', 3: '#31fc02'}  # Red, Blue, Purple, Green
    areas = np.array([0.74**2, 0.46**2, (1.37*0.8)**2, 0.99**2]) * 500
    size_map = {0: areas[0], 1: areas[1], 2: areas[2], 3: areas[3]}
    
    node_colors = [color_map[node_type_ids[i]] for i in G.nodes()]
    node_sizes = [size_map[node_type_ids[i]] for i in G.nodes()]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, 
                          edge_color='gray', 
                          width=3, 
                          alpha=0.6,
                          ax=ax)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos,
                          node_color=node_colors,
                          node_size=node_sizes,
                          edgecolors='#333333',
                          linewidths=1.5,
                          ax=ax)
    
    
    # Styling
    ax.set_aspect('equal')
    ax.axis('off')
    ax.margins(0.05)
    
    plt.tight_layout()
    return fig, ax

In [ ]:
fig, ax = plot_2d_graph_for_workflow(
    g, 
    structure,
    figsize=(8, 8)
)
# plt.savefig('./data/graph_representation-structure-%s.png'%sp_idx, dpi=200, bbox_inches='tight')
plt.show()

### 5. PDOS and Energy-resolved Node attribution

In [ ]:
from pathlib import Path

In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor

from pymatgen.core.periodic_table import Element
from pymatgen.electronic_structure.core import OrbitalType
from pymatgen.io.vasp.outputs import Vasprun

from scipy.interpolate import interp1d

In [ ]:
structures = pickle.load(open(ROOT_PATH + '/dataset/all_structures.pkl', 'rb'))
spectra = torch.load(ROOT_PATH + "/dataset/all_spectra.pt")
shifts = np.load('./data/sp_shifts.npy')  ## Delta_SCF shift for spectra

In [ ]:
Element_list = [Element[symbol] for symbol in element_types]
orbital_symbol, orbital = 'p', OrbitalType.p

cluster_reps = [1955, 923, 1878, 1210, 1858, 976]

Epdos_grid = np.linspace(-10, 20, 3001)

In [ ]:
dict_pred_rep_sp = {}
for cluster_idx in range(1): 
    dict_pred_rep_sp[cluster_idx] = np.loadtxt("./data/rep_spec-pred-cluster_%s.txt"%(cluster_idx), unpack=True)[1]

In [ ]:
loadpath = ROOT_PATH + '/data/data-PDOS/'
fn_vasprun = 'vasprun.xml'

In [ ]:
dict_pdos_attr = {}
for cluster_idx in range(1): 
    sp_idx = cluster_reps[cluster_idx]
    structure = structures[sp_idx]
    sp_dft = spectra[sp_idx].numpy()
    sp_pred = dict_pred_rep_sp[cluster_idx]
    node_importance_0baseline_arr = np.loadtxt('./data/data-IG/0_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff))
    node_importance_OCl_arr = np.loadtxt('./data/data-IG/OCl_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff))
    
    pdos_path = loadpath + 'structure-%s'%(sp_idx)
    print(pdos_path)
    vasprun = Vasprun(Path(pdos_path) / Path(fn_vasprun), parse_dos = True)
    complete_dos = vasprun.complete_dos    
    
    atoms = AseAtomsAdaptor.get_atoms(structure)
    first_neighbors = structure.get_neighbors(structure[0], r=r1)  # Using 3.0 Angstrom cutoff
    first_neighbors = [neighbor for neighbor in first_neighbors if neighbor.specie.symbol != 'H']
    first_neighbor_species = [neighbor.specie.symbol for neighbor in first_neighbors]    

    idxs_Cl = []
    idxs_O = []
    for site in first_neighbors: 
        if site.specie.symbol == 'Cl': 
            idxs_Cl.append(site.index)
        if site.specie.symbol == 'O': 
            idxs_O.append(site.index)
    idxs_Cl.sort()        
    idxs_O.sort()         

    dict_pdos_attr[cluster_idx] = {
        'shift_pdos': shifts[sp_idx] - np.mean(shifts), 
        'sp_dft': sp_dft, 
        'sp_pred': sp_pred, 
        'efermi': complete_dos.efermi,
    }    

    spd_dos = complete_dos.get_site_spd_dos(structure[0])
    dos = spd_dos[orbital]
    dict_pdos_attr[cluster_idx]['pdos_Zn'] = [dos.energies, dos.get_densities()]    
    if len(idxs_O) > 0: 
        dos_O = []
        for site_idx in idxs_O: 
            spd_dos = complete_dos.get_site_spd_dos(structure[site_idx])
            dos = spd_dos[orbital]
            dos_O.append(dos.get_densities()) 
        dict_pdos_attr[cluster_idx]['pdos_O'] = [complete_dos.energies, np.mean(dos_O, axis=0)]
    if len(idxs_Cl) > 0: 
        dos_Cl = []
        for site_idx in idxs_Cl: 
            spd_dos = complete_dos.get_site_spd_dos(structure[site_idx])
            dos = spd_dos[orbital]
            dos_Cl.append(dos.get_densities()) 
        dict_pdos_attr[cluster_idx]['pdos_Cl'] = [complete_dos.energies, np.mean(dos_Cl, axis=0)]      
    
    dict_pdos_attr[cluster_idx]['attr_Zn'] = node_importance_0baseline_arr[:,0]/np.max(node_importance_0baseline_arr)
    if len(idxs_O) > 0: 
        dict_pdos_attr[cluster_idx]['attr_O'] = node_importance_0baseline_arr[:,idxs_O]/np.max(node_importance_0baseline_arr)
        dict_pdos_attr[cluster_idx]['attr_cf_O'] = node_importance_OCl_arr[:,idxs_O]/np.max(abs(node_importance_OCl_arr[:,1:]))
        
    if len(idxs_Cl) > 0: 
        dict_pdos_attr[cluster_idx]['attr_Cl'] = node_importance_0baseline_arr[:,idxs_Cl]/np.max(node_importance_0baseline_arr)       
        dict_pdos_attr[cluster_idx]['attr_cf_Cl'] = node_importance_OCl_arr[:,idxs_Cl]/np.max(abs(node_importance_OCl_arr[:,1:]))

In [ ]:
# pickle.dump(dict_pdos_attr, open('./data/dict_pdos_attr.pkl', 'wb'), protocol=4)

### 6. Edge attribution

In [ ]:
dict_edge_attr = {}
delta = 0.05

In [ ]:
for cluster_idx in range(1): 
    sp_idx = cluster_reps[cluster_idx]
    structure = structures[sp_idx]
    sp_dft = spectra[sp_idx].numpy()
    sp_pred = dict_pred_rep_sp[cluster_idx]
    edge_importance_arr = np.loadtxt('./data/data-IG/%s-importance_bond-structure_%s-%s-%s-%s.txt'%(delta, sp_idx, nblocks, cutoff, threebody_cutoff))

    
    g = all_data["graphs"][sp_idx]
    src, dst = g.edges()
    bond_dist = g.edata["bond_dist"].numpy().flatten()
    node_types = g.ndata["node_type"].numpy()
    
    fs_bonds = []
    mask_in = (dst == 0)
    for bond_idx in range(len(bond_dist)): 
        if mask_in[bond_idx]: 
            if node_types[src[bond_idx]] == 1: # ignore H atom
                continue
            fs_bonds.append(
                (bond_idx, src[bond_idx].item())
            )
    fs_bonds.sort(key=lambda x: x[1])
    
    dict_edge_attr[cluster_idx] = {
        'sp_dft': sp_dft, 
        'sp_pred': sp_pred, 
    }        
    dict_edge_attr[cluster_idx]['attr'] = []
    dict_edge_attr[cluster_idx]['element'] = []
    dict_edge_attr[cluster_idx]['dist'] = []
    for bond_idx, src_idx in fs_bonds: 
        element_symbol, bond_len = element_types[node_types[src_idx]], bond_dist[bond_idx]
        if bond_len < r1: 
            dict_edge_attr[cluster_idx]['attr'].append(edge_importance_arr[:, bond_idx]/np.max(abs(edge_importance_arr)) * (-delta / abs(delta)))
            dict_edge_attr[cluster_idx]['element'].append(element_symbol)
            dict_edge_attr[cluster_idx]['dist'].append(bond_len)
    dict_edge_attr[cluster_idx]['attr'] = np.array(dict_edge_attr[cluster_idx]['attr'])
    dict_edge_attr[cluster_idx]['dist'] = np.array(dict_edge_attr[cluster_idx]['dist'])

In [ ]:
# pickle.dump(dict_edge_attr, open('./data/dict_edge_attr-0.07.pkl', 'wb'), protocol=4)